# InceptionV3 — Garlic Disease Classification (Multi-Run Experiment)

| Item | Value |
|---|---|
| **Architecture** | InceptionV3 (ImageNet pretrained, partial fine-tune) |
| **Input size** | 299 × 299 × 3 |
| **Strategy** | 3-seed independent runs for statistical robustness |
| **Optimizer** | Adam + ExponentialDecay (lr=1e-5) |
| **Loss** | CategoricalCrossentropy (label_smoothing=0.15) |
| **Regularisation** | Dropout 0.5 + L2 1e-5 + Class-weight balancing |

---
### InceptionV3 Unfreeze Strategy
```
Total layers : ~311, organized into Inception blocks mixed0 – mixed10
Frozen       : mixed0 – mixed6  (low-level: edges, textures, colors)
Trainable    : mixed7 – mixed10 (high-level: complex patterns, ~14M / 23M params)
BN layers    : always frozen  → prevent covariate shift with small dataset
```
---


In [ ]:
# ========== 1. IMPORTS ========== #
import os
import glob
import time
import random
import shutil
import tempfile
from collections import defaultdict
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm_lib
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.regularizers import l2

from sklearn.utils import class_weight
from sklearn.metrics import (classification_report, confusion_matrix,
                              top_k_accuracy_score, roc_curve, auc,
                              cohen_kappa_score, matthews_corrcoef,
                              balanced_accuracy_score)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ========== 2. GPU CONFIG ========== #
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", len(gpus))
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as e:
        print(e)

# ========== 3. MIXED PRECISION ========== #
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Compute dtype :", tf.keras.mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", tf.keras.mixed_precision.global_policy().variable_dtype)


In [ ]:
# ========== 4. EXPERIMENT CONFIGURATION ========== #

# --- Paths ---
DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
BASE_RESULT_DIR = "/kaggle/working/report_InceptionV3_MultiRun"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# --- Model ---
INPUT_SHAPE = (299, 299, 3)   # InceptionV3 native resolution

BATCH_SIZE  = 64
EPOCHS      = 30

# --- InceptionV3 unfreeze strategy ---
# 311 total layers, organized into Inception blocks mixed0–mixed10
# Freeze  : mixed0–mixed6  (low-level: edges, textures, colors)
# Unfreeze: mixed7–mixed10 (high-level: complex patterns, ~14M / 23M params)
# BN layers: always frozen → prevent covariate shift with small dataset
INCEPTION_UNFREEZE_LAYERS = ['mixed7', 'mixed8', 'mixed9', 'mixed10']

# --- Multi-run settings ---
RANDOM_SEEDS = [42, 123, 456]

# --- Performance knobs ---
AUTOTUNE = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)   # XLA JIT → ~15-30% GPU throughput gain

# --- Result storage ---
all_runs_results = []

print(f"Dataset    : {DATA_DIR}")
print(f"Output dir : {BASE_RESULT_DIR}")
print(f"Input shape: {INPUT_SHAPE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Seeds      : {RANDOM_SEEDS}")
print(f"XLA JIT    : ON")
print(f"\nInceptionV3 unfreeze blocks: {INCEPTION_UNFREEZE_LAYERS}")
print(f"  → Frozen  : mixed0 – mixed6  (low-level features)")
print(f"  → Trainable: mixed7 – mixed10 (~14M / 23M params)")
print(f"  → BN layers: always frozen")


In [ ]:
# ========== 5. HELPER FUNCTIONS ========== #

_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.083),        # ≈ ±30°
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
], name='augmentation')


def _collect_samples(split_dir, class_to_idx):
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None):
    class_names  = sorted([
        d for d in os.listdir(os.path.join(data_dir, 'train'))
        if os.path.isdir(os.path.join(data_dir, 'train', d))
    ])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes  = len(class_names)
    h, w         = input_shape[:2]

    def load_and_preprocess(path, label):
        img   = tf.image.decode_jpeg(tf.io.read_file(path), channels=3)
        img   = tf.image.resize(img, [h, w])
        img   = inception_preprocess(tf.cast(img, tf.float32))  # scales to [-1, 1]
        label = tf.one_hot(label, depth=num_classes)
        return img, label

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False):
        sdir               = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        n  = len(paths)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(n, seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if training:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training)
        ds = ds.prefetch(AUTOTUNE)
        return ds, n, fns, labels

    train_ds, n_train, _,           train_lbl  = _make_split('train', training=True)
    val_ds,   n_val,   _,           _          = _make_split('val',   training=False)
    test_ds,  n_test,  test_fnames, test_lbl   = _make_split('test',  training=False)

    cw_dict = dict(enumerate(
        class_weight.compute_class_weight(
            'balanced', classes=np.unique(train_lbl), y=train_lbl)))

    meta = SimpleNamespace(
        class_names       = class_names,
        num_classes       = num_classes,
        test_filenames    = test_fnames,
        test_classes      = np.array(test_lbl),
        n_train           = n_train,
        n_val             = n_val,
        n_test            = n_test,
        class_weight_dict = cw_dict,
    )
    return train_ds, val_ds, test_ds, meta


def build_model(input_shape, num_classes, steps_per_epoch):
    """Build InceptionV3 with partial fine-tuning + custom head.

    Unfreeze strategy:
      - Freeze entire backbone first
      - Re-enable trainable on layers belonging to mixed7–mixed10
      - BN layers are always kept frozen (trainable=False)
    """
    base = InceptionV3(weights='imagenet', include_top=False, input_shape=input_shape)

    # Step 1: freeze everything
    base.trainable = False

    # Step 2: selectively unfreeze mixed7–mixed10 (non-BN layers only)
    for layer in base.layers:
        if any(layer.name.startswith(block) for block in INCEPTION_UNFREEZE_LAYERS):
            if not isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = True

    trainable_count     = sum(1 for l in base.layers if l.trainable)
    non_trainable_count = sum(1 for l in base.layers if not l.trainable)
    print(f"  Backbone trainable layers    : {trainable_count}/{len(base.layers)}")
    print(f"  Backbone non-trainable layers: {non_trainable_count}/{len(base.layers)}")

    # Custom classification head
    x   = GlobalAveragePooling2D()(base.output)       # 2048-d
    x   = BatchNormalization()(x)
    x   = Dense(128, activation='relu', kernel_regularizer=l2(1e-5))(x)
    x   = Dropout(0.5)(x)
    out = Dense(num_classes, activation='softmax', dtype='float32')(x)
    model = Model(inputs=base.input, outputs=out)

    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=1e-5,
        decay_steps=steps_per_epoch * 5,
        decay_rate=0.9,
        staircase=True,
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss=CategoricalCrossentropy(label_smoothing=0.15),
        metrics=['accuracy'],
    )
    return model


print("Helper functions defined.")
print(f"  create_tf_datasets — tf.data pipeline (GPU-optimised)")
print(f"  build_model        — InceptionV3 (mixed7–mixed10 unfrozen) + custom head")


In [ ]:
# ========== 6. MULTI-RUN TRAINING EXPERIMENT ========== #
for run_idx, seed in enumerate(RANDOM_SEEDS):
    print("\n" + "="*70)
    print(f" RUN {run_idx+1}/{len(RANDOM_SEEDS)}  —  seed={seed}")
    print("="*70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    # --- tf.data pipeline ---
    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed)

    steps_per_epoch = meta.n_train // BATCH_SIZE

    # --- Build model (InceptionV3 + partial fine-tune) ---
    print(f"\n  Building InceptionV3 model ...")
    model = build_model(INPUT_SHAPE, meta.num_classes, steps_per_epoch)

    total_params     = model.count_params()
    trainable_params = sum(tf.size(w).numpy() for w in model.trainable_weights)
    print(f"  Total params    : {total_params:,}")
    print(f"  Trainable params: {trainable_params:,}")

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10,
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv'), append=False),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'inceptionv3_best.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ]

    # --- Train ---
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=meta.class_weight_dict,
        callbacks=callbacks,
    )

    # --- Learning curves ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, keys, title in zip(axes,
                                [('accuracy', 'val_accuracy'), ('loss', 'val_loss')],
                                ['Accuracy', 'Loss']):
        ax.plot(history.history[keys[0]], label='Train')
        ax.plot(history.history[keys[1]], label='Validation')
        ax.set_title(f'{title} — Run {run_idx+1} (seed={seed})')
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'learning_curve.png'), dpi=300)
    plt.close()

    # --- Evaluation ---
    best_model = load_model(os.path.join(RESULT_DIR, 'inceptionv3_best.keras'))
    pred_probs = best_model.predict(test_ds, verbose=1)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes
    class_names = meta.class_names

    report = classification_report(y_true_run, y_pred_run,
                                   target_names=class_names, output_dict=True, digits=4)
    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(y_true_run, y_pred_run,
                                      target_names=class_names, digits=4))

    # --- Confusion matrix ---
    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=class_names, yticklabels=class_names, cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — Run {run_idx+1} (seed={seed})')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.close()

    # --- Store run results ---
    test_acc = np.mean(y_pred_run == y_true_run)
    all_runs_results.append({
        'run':            run_idx + 1,
        'seed':           seed,
        'accuracy':       test_acc,
        'precision':      report['weighted avg']['precision'],
        'recall':         report['weighted avg']['recall'],
        'f1_score':       report['weighted avg']['f1-score'],
        'per_class_metrics': {
            c: {'precision': report[c]['precision'],
                'recall':    report[c]['recall'],
                'f1-score':  report[c]['f1-score']}
            for c in class_names
        },
        'result_dir':     RESULT_DIR,
        'history':        history.history,
        'y_true':         y_true_run,
        'y_pred':         y_pred_run,
        'pred_probs':     pred_probs,
        'class_names':    class_names,
        'test_filenames': meta.test_filenames,
        'n_train':        meta.n_train,
        'n_val':          meta.n_val,
        'n_test':         meta.n_test,
    })

    print(f"\n  Acc={test_acc:.4f}  P={report['weighted avg']['precision']:.4f}"
          f"  R={report['weighted avg']['recall']:.4f}"
          f"  F1={report['weighted avg']['f1-score']:.4f}")

    tf.keras.backend.clear_session()

print("\n" + "="*70)
print(" ALL TRAINING RUNS COMPLETED")
print("="*70)


---
## Section 2 — Results Aggregation & Scientific Reports

Aggregate metrics across all runs, generate LaTeX tables, CSV summaries, and visualizations for publication.


In [ ]:
# ========== AGGREGATE RESULTS FROM ALL RUNS ========== #
accuracies = [r['accuracy']  for r in all_runs_results]
precisions = [r['precision'] for r in all_runs_results]
recalls    = [r['recall']    for r in all_runs_results]
f1_scores  = [r['f1_score']  for r in all_runs_results]

overall_stats = {
    'Accuracy':  {'mean': np.mean(accuracies), 'std': np.std(accuracies), 'values': accuracies},
    'Precision': {'mean': np.mean(precisions), 'std': np.std(precisions), 'values': precisions},
    'Recall':    {'mean': np.mean(recalls),    'std': np.std(recalls),    'values': recalls},
    'F1-Score':  {'mean': np.mean(f1_scores),  'std': np.std(f1_scores),  'values': f1_scores},
}

print("OVERALL METRICS ACROSS ALL RUNS:")
print("-" * 80)
for metric_name, stats in overall_stats.items():
    print(f"{metric_name:12s}: {stats['mean']:.4f} ± {stats['std']:.4f}")
    print(f"              Individual runs: {[f'{v:.4f}' for v in stats['values']]}")
print("-" * 80)

class_names = list(all_runs_results[0]['per_class_metrics'].keys())

per_class_stats = {}
for class_name in class_names:
    per_class_stats[class_name] = {}
    for metric in ['precision', 'recall', 'f1-score']:
        values = [r['per_class_metrics'][class_name][metric] for r in all_runs_results]
        per_class_stats[class_name][metric] = {
            'mean': np.mean(values), 'std': np.std(values), 'values': values}

print("\nStatistics calculated successfully!")


In [ ]:
# ========== CREATE SCIENTIFIC REPORT TABLES ========== #
overall_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Mean':   [overall_stats[m]['mean'] for m in ['Accuracy','Precision','Recall','F1-Score']],
    'Std':    [overall_stats[m]['std']  for m in ['Accuracy','Precision','Recall','F1-Score']],
    'Run 1':  [accuracies[0], precisions[0], recalls[0], f1_scores[0]],
    'Run 2':  [accuracies[1], precisions[1], recalls[1], f1_scores[1]],
    'Run 3':  [accuracies[2], precisions[2], recalls[2], f1_scores[2]],
})
overall_df['Mean ± Std'] = overall_df.apply(
    lambda row: f"{row['Mean']:.4f} ± {row['Std']:.4f}", axis=1)

print("OVERALL PERFORMANCE METRICS (3 RUNS)")
print("="*80)
print(overall_df[['Metric', 'Mean ± Std', 'Run 1', 'Run 2', 'Run 3']].to_string(index=False))
overall_df.to_csv(os.path.join(BASE_RESULT_DIR, "overall_metrics_summary.csv"), index=False)

per_class_rows = []
for class_name in class_names:
    for metric in ['precision', 'recall', 'f1-score']:
        stats = per_class_stats[class_name][metric]
        per_class_rows.append({
            'Class': class_name, 'Metric': metric.capitalize(),
            'Mean': stats['mean'], 'Std': stats['std'],
            'Mean ± Std': f"{stats['mean']:.4f} ± {stats['std']:.4f}",
            'Run 1': stats['values'][0], 'Run 2': stats['values'][1], 'Run 3': stats['values'][2],
        })
per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(os.path.join(BASE_RESULT_DIR, "per_class_metrics_summary.csv"), index=False)

print("\nPER-CLASS PERFORMANCE METRICS (3 RUNS)")
for class_name in class_names:
    print(f"\n{class_name}:")
    print(per_class_df[per_class_df['Class'] == class_name][['Metric', 'Mean ± Std']].to_string(index=False))

print("\nSummary tables saved to CSV files!")


In [ ]:
# ========== GENERATE LATEX TABLE FOR PAPER ========== #
latex_overall = r"""\begin{table}[h]
\centering
\caption{Overall Performance Metrics of InceptionV3 (Mean $\pm$ Std over 3 runs)}
\label{tab:inceptionv3_overall}
\begin{tabular}{lcccc}
\hline
\textbf{Metric} & \textbf{Mean $\pm$ Std} & \textbf{Run 1} & \textbf{Run 2} & \textbf{Run 3} \\
\hline
"""
for _, row in overall_df.iterrows():
    latex_overall += f"{row['Metric']} & {row['Mean ± Std']} & {row['Run 1']:.4f} & {row['Run 2']:.4f} & {row['Run 3']:.4f} \\\\\n"
latex_overall += r"""\hline
\end{tabular}
\end{table}
"""

latex_per_class = r"""\begin{table}[h]
\centering
\caption{Per-Class Performance Metrics of InceptionV3 (Mean $\pm$ Std over 3 runs)}
\label{tab:inceptionv3_per_class}
\begin{tabular}{lccc}
\hline
\textbf{Class} & \textbf{Precision} & \textbf{Recall} & \textbf{F1-Score} \\
\hline
"""
for class_name in class_names:
    prec = per_class_stats[class_name]['precision']
    rec  = per_class_stats[class_name]['recall']
    f1   = per_class_stats[class_name]['f1-score']
    latex_per_class += (f"{class_name} & {prec['mean']:.4f} $\\pm$ {prec['std']:.4f} & "
                        f"{rec['mean']:.4f} $\\pm$ {rec['std']:.4f} & "
                        f"{f1['mean']:.4f} $\\pm$ {f1['std']:.4f} \\\\\n")
latex_per_class += r"""\hline
\end{tabular}
\end{table}
"""

print(latex_overall)
print(latex_per_class)

with open(os.path.join(BASE_RESULT_DIR, "latex_tables.tex"), "w") as f:
    f.write("% Overall Metrics Table\n"); f.write(latex_overall)
    f.write("\n\n% Per-Class Metrics Table\n"); f.write(latex_per_class)
print("LaTeX tables saved to 'latex_tables.tex'")


In [ ]:
# ========== VISUALIZATION OF RESULTS ACROSS RUNS ========== #
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
means = [overall_stats[m]['mean'] for m in metrics_list]
stds  = [overall_stats[m]['std']  for m in metrics_list]
x_pos = np.arange(len(metrics_list))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bar plot with error bars
axes[0].bar(x_pos, means, yerr=stds, capsize=10, alpha=0.8,
            color='steelblue', edgecolor='black')
axes[0].set_title('InceptionV3 Performance (Mean ± Std, 3 runs)', fontweight='bold')
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(metrics_list)
axes[0].set_ylim([0, 1.05]); axes[0].grid(axis='y', alpha=0.3)
for i, (m, s) in enumerate(zip(means, stds)):
    axes[0].text(i, m + s + 0.02, f'{m:.4f}\n±{s:.4f}', ha='center', fontsize=8, fontweight='bold')

# Box plot
bp = axes[1].boxplot([accuracies, precisions, recalls, f1_scores],
                     labels=metrics_list, patch_artist=True, showmeans=True,
                     meanprops=dict(marker='D', markerfacecolor='red', markersize=8))
for patch in bp['boxes']:
    patch.set_facecolor('lightblue'); patch.set_alpha(0.7)
axes[1].set_title('Distribution Across 3 Runs', fontweight='bold')
axes[1].set_ylim([0, 1.05]); axes[1].grid(axis='y', alpha=0.3)

# Per-class F1
class_f1_means = [per_class_stats[c]['f1-score']['mean'] for c in class_names]
class_f1_stds  = [per_class_stats[c]['f1-score']['std']  for c in class_names]
axes[2].bar(np.arange(len(class_names)), class_f1_means, yerr=class_f1_stds,
            capsize=5, alpha=0.8, color='coral', edgecolor='black')
axes[2].set_title('Per-Class F1-Score (Mean ± Std)', fontweight='bold')
axes[2].set_xticks(np.arange(len(class_names)))
axes[2].set_xticklabels(class_names, rotation=45, ha='right')
axes[2].set_ylim([0, 1.05]); axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('InceptionV3 Multi-Run Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "metrics_visualization.png"), dpi=300, bbox_inches='tight')
plt.show()
print("Visualizations saved!")


In [ ]:
# ========== COMPREHENSIVE MULTI-RUN SUMMARY REPORT ========== #
report_lines = [
    "=" * 70,
    "InceptionV3 - MULTI-RUN TRAINING REPORT",
    "=" * 70,
    f"Seeds: {SEEDS}",
    f"Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}  |  LR: {LEARNING_RATE}",
    f"Unfreeze Blocks: {INCEPTION_UNFREEZE_LAYERS}",
    f"Label Smoothing: {LABEL_SMOOTHING}  |  Dropout: 0.5",
    f"Input Shape: {INPUT_SHAPE}",
    "",
    "─" * 70,
    "OVERALL METRICS (Mean ± Std over 3 seeds)",
    "─" * 70,
]
for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    report_lines.append(f"  {m:<15}: {overall_stats[m]['mean']:.4f} ± {overall_stats[m]['std']:.4f}"
                        f"  (min={overall_stats[m]['min']:.4f}, max={overall_stats[m]['max']:.4f})")
report_lines += ["", "─" * 70, "PER-CLASS METRICS", "─" * 70]
for c in class_names:
    report_lines.append(f"  {c}:")
    for metric in ['precision', 'recall', 'f1-score']:
        s = per_class_stats[c][metric]
        report_lines.append(f"    {metric:<12}: {s['mean']:.4f} ± {s['std']:.4f}")
report_lines += ["", "─" * 70, "PER-RUN RESULTS", "─" * 70]
for rd in all_run_data:
    report_lines.append(f"  Seed {rd['seed']:>3}: Acc={rd['accuracy']:.4f}  P={rd['precision']:.4f}  R={rd['recall']:.4f}  F1={rd['f1']:.4f}")
report_lines.append("=" * 70)

report_text = "\n".join(report_lines)
print(report_text)
report_path = os.path.join(BASE_RESULT_DIR, "MULTI_RUN_SUMMARY_REPORT.txt")
with open(report_path, "w") as f:
    f.write(report_text)
print(f"\nReport saved → {report_path}")


In [ ]:
# ========== ZIP MULTI-RUN RESULTS ========== #
import shutil
zip_output_path = "/kaggle/working/InceptionV3_MultiRun_Complete"
shutil.make_archive(zip_output_path, 'zip', BASE_RESULT_DIR)
print(f"Multi-run results zipped → {zip_output_path}.zip")
from IPython.display import FileLink
display(FileLink(f'{zip_output_path}.zip'))


# Section 3 — Deep Diagnostic Analysis

This section performs in-depth analysis using the best single run (selected by highest F1-score):
- Dataset distribution
- Aggregate confusion matrix (all 3 runs)
- ROC curves & AUC
- Advanced metrics (Cohen's κ, MCC, Balanced Accuracy)
- Convergence analysis
- Grad-CAM saliency maps
- t-SNE feature embedding
- Error analysis & qualitative review


In [ ]:
# ========== DATASET DISTRIBUTION ANALYSIS ========== #
import glob as _glob

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
split_names = ['Train', 'Validation', 'Test']
split_dirs  = [TRAIN_DIR, VAL_DIR, TEST_DIR]
colors = ['steelblue', 'darkorange', 'seagreen']

all_counts = []
for ax, split, sdir, color in zip(axes, split_names, split_dirs, colors):
    counts = []
    for cls in class_names:
        n = len(_glob.glob(os.path.join(sdir, cls, '*')))
        counts.append(n)
    all_counts.append(counts)
    bars = ax.bar(class_names, counts, color=color, alpha=0.8, edgecolor='black')
    ax.set_title(f'{split} Split (total={sum(counts)})', fontweight='bold')
    ax.set_ylim([0, max(counts) * 1.25])
    ax.set_xticklabels(class_names, rotation=30, ha='right')
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.02,
                str(cnt), ha='center', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Dataset Distribution — InceptionV3 Experiment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "dataset_distribution.png"), dpi=300, bbox_inches='tight')
plt.show()
print("Dataset distribution saved!")


In [ ]:
# ========== AGGREGATE CONFUSION MATRIX (ALL RUNS) ========== #
from sklearn.metrics import confusion_matrix as sk_cm

agg_cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
for rd in all_run_data:
    agg_cm += sk_cm(rd['y_true'], rd['y_pred'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(agg_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names)
axes[0].set_title('Aggregate CM — Raw Counts (3 runs)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# Normalised
agg_cm_norm = agg_cm.astype(float) / agg_cm.sum(axis=1, keepdims=True)
sns.heatmap(agg_cm_norm, annot=True, fmt='.3f', cmap='Blues', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title('Aggregate CM — Normalised (3 runs)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

plt.suptitle('Confusion Matrices — InceptionV3 (Aggregate over 3 runs)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "aggregate_confusion_matrix.png"), dpi=300, bbox_inches='tight')
plt.show()
print("Aggregate confusion matrix saved!")


In [ ]:
# ========== ROC CURVES (AGGREGATE OVER ALL RUNS) ========== #
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

all_y_true_bin = []
all_y_prob     = []
for rd in all_run_data:
    yb = label_binarize(rd['y_true'], classes=list(range(NUM_CLASSES)))
    all_y_true_bin.append(yb)
    all_y_prob.append(np.array(rd['y_prob']))

y_true_agg = np.vstack(all_y_true_bin)
y_prob_agg = np.vstack(all_y_prob)

fig, ax = plt.subplots(figsize=(8, 6))
palette = ['steelblue', 'darkorange', 'seagreen']
for i, (cls, col) in enumerate(zip(class_names, palette)):
    fpr, tpr, _ = roc_curve(y_true_agg[:, i], y_prob_agg[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=col, lw=2, label=f'{cls} (AUC={roc_auc:.4f})')

ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — InceptionV3 (Aggregate over all runs)', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "roc_curves.png"), dpi=300, bbox_inches='tight')
plt.show()
print("ROC curves saved!")


In [ ]:
# ========== ADDITIONAL METRICS: COHEN'S KAPPA, MCC, BALANCED ACCURACY ========== #
from sklearn.metrics import cohen_kappa_score, matthews_corrcoef, balanced_accuracy_score

print("=" * 60)
print("ADDITIONAL METRICS PER RUN")
print("=" * 60)
kappas, mccs, bal_accs = [], [], []
for rd in all_run_data:
    k   = cohen_kappa_score(rd['y_true'], rd['y_pred'])
    mcc = matthews_corrcoef(rd['y_true'], rd['y_pred'])
    ba  = balanced_accuracy_score(rd['y_true'], rd['y_pred'])
    kappas.append(k); mccs.append(mcc); bal_accs.append(ba)
    print(f"  Seed {rd['seed']:>3}: κ={k:.4f}  MCC={mcc:.4f}  BalAcc={ba:.4f}")

print("─" * 60)
print(f"  Mean κ={np.mean(kappas):.4f}±{np.std(kappas):.4f}  "
      f"MCC={np.mean(mccs):.4f}±{np.std(mccs):.4f}  "
      f"BalAcc={np.mean(bal_accs):.4f}±{np.std(bal_accs):.4f}")
print("=" * 60)

adv_df = pd.DataFrame({
    'Seed': [rd['seed'] for rd in all_run_data],
    'Cohen_Kappa': kappas, 'MCC': mccs, 'Balanced_Accuracy': bal_accs
})
adv_df.to_csv(os.path.join(BASE_RESULT_DIR, "advanced_metrics.csv"), index=False)
print("Advanced metrics saved!")


In [ ]:
# ========== CONVERGENCE ANALYSIS (ALL RUNS) ========== #
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette_run = ['steelblue', 'darkorange', 'seagreen']

for rd, col in zip(all_run_data, palette_run):
    log_path = os.path.join(rd['result_dir'], 'training_log.csv')
    if not os.path.exists(log_path):
        continue
    log_df = pd.read_csv(log_path)
    label  = f"Seed {rd['seed']}"
    axes[0].plot(log_df['epoch'], log_df['loss'],     color=col, linestyle='-',  label=f'{label} train')
    axes[0].plot(log_df['epoch'], log_df['val_loss'], color=col, linestyle='--', label=f'{label} val')
    axes[1].plot(log_df['epoch'], log_df['accuracy'],     color=col, linestyle='-',  label=f'{label} train')
    axes[1].plot(log_df['epoch'], log_df['val_accuracy'], color=col, linestyle='--', label=f'{label} val')

axes[0].set_title('Loss Convergence', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)

axes[1].set_title('Accuracy Convergence', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)

plt.suptitle('Training Convergence Analysis — InceptionV3', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "convergence_analysis.png"), dpi=300, bbox_inches='tight')
plt.show()
print("Convergence analysis saved!")


In [ ]:
# ========== SELECT BEST RUN FOR SINGLE-RUN ANALYSIS ========== #
best_run   = max(all_run_data, key=lambda x: x['f1'])
SELECTED_RUN = best_run['seed']
RESULT_DIR   = best_run['result_dir']
print(f"Selected Run: seed={SELECTED_RUN}  F1={best_run['f1']:.4f}  Acc={best_run['accuracy']:.4f}")
print(f"Result dir: {RESULT_DIR}")

# Reload model
from tensorflow.keras.models import load_model
model = load_model(os.path.join(RESULT_DIR, 'inceptionv3_best.keras'))
print("Model loaded successfully!")

# Rebuild test dataset (deterministic, no shuffle)
_, _, test_ds, test_filenames, test_labels_true = create_tf_datasets(
    TRAIN_DIR, VAL_DIR, TEST_DIR, BATCH_SIZE, INPUT_SHAPE[:2], seed=SELECTED_RUN
)
print(f"Test samples: {len(test_filenames)}")


In [ ]:
# ========== GRAD-CAM VISUALIZATION ========== #
import cv2

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads   = tape.gradient(class_channel, conv_outputs)
    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out= conv_outputs[0]
    heatmap = conv_out @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

# InceptionV3 last conv layer
LAST_CONV = 'mixed10'

n_show = min(8, len(test_filenames))
fig, axes = plt.subplots(n_show, 2, figsize=(8, n_show * 2.5))

for i in range(n_show):
    img_path = test_filenames[i]
    img_pil  = tf.keras.preprocessing.image.load_img(img_path, target_size=INPUT_SHAPE[:2])
    img_arr  = tf.keras.preprocessing.image.img_to_array(img_pil)
    img_proc = inception_preprocess(np.expand_dims(img_arr.copy(), 0))
    preds    = model.predict(img_proc, verbose=0)
    pred_cls = np.argmax(preds[0])
    true_cls = test_labels_true[i]

    heatmap = make_gradcam_heatmap(img_proc, model, LAST_CONV, pred_cls)
    heatmap_resized = cv2.resize(heatmap, INPUT_SHAPE[:2][::-1])
    heatmap_colored = cv2.applyColorMap(np.uint8(255*heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    superimposed  = heatmap_colored * 0.4 + img_arr
    superimposed  = np.clip(superimposed / superimposed.max(), 0, 1)

    axes[i, 0].imshow(img_arr.astype('uint8'))
    axes[i, 0].set_title(f'True: {class_names[true_cls]}', fontsize=7)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(superimposed)
    status = '✓' if pred_cls == true_cls else '✗'
    axes[i, 1].set_title(f'Pred: {class_names[pred_cls]} {status}', fontsize=7)
    axes[i, 1].axis('off')

plt.suptitle(f'Grad-CAM — InceptionV3 (Run {SELECTED_RUN})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, "gradcam_visualization.png"), dpi=200, bbox_inches='tight')
plt.show()
print("Grad-CAM visualization saved!")


In [ ]:
# ========== t-SNE FEATURE EMBEDDING ========== #
from sklearn.manifold import TSNE

# Extract features from the GAP layer (before final dense)
feature_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer('global_average_pooling2d').output
)

all_features, all_true_labels = [], []
for imgs, lbls in test_ds:
    feats = feature_model.predict(imgs, verbose=0)
    all_features.append(feats)
    all_true_labels.extend(lbls.numpy())

features_np = np.vstack(all_features)
labels_np   = np.array(all_true_labels)

print(f"Feature matrix shape: {features_np.shape}")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
emb  = tsne.fit_transform(features_np)

fig, ax = plt.subplots(figsize=(9, 7))
palette = ['steelblue', 'darkorange', 'seagreen']
for i, (cls, col) in enumerate(zip(class_names, palette)):
    mask = labels_np == i
    ax.scatter(emb[mask, 0], emb[mask, 1], c=col, label=cls, alpha=0.7, s=40, edgecolors='white', linewidths=0.5)

ax.set_title(f't-SNE Feature Embedding — InceptionV3 (Run {SELECTED_RUN})', fontweight='bold')
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, "tsne_embedding.png"), dpi=300, bbox_inches='tight')
plt.show()
print("t-SNE plot saved!")


In [ ]:
# ========== ERROR ANALYSIS ========== #
y_pred_run = best_run['y_pred']
y_true_run = best_run['y_true']
y_prob_run = np.array(best_run['y_prob'])

wrong_idx = np.where(np.array(y_pred_run) != np.array(y_true_run))[0]
print(f"Total misclassifications: {len(wrong_idx)} / {len(y_true_run)}")

# Confidence of wrong predictions
wrong_conf = y_prob_run[wrong_idx, np.array(y_pred_run)[wrong_idx]]
print(f"Mean confidence on errors: {wrong_conf.mean():.4f}  Max: {wrong_conf.max():.4f}")

# Confusion pairs
from collections import Counter
error_pairs = Counter()
for idx in wrong_idx:
    pair = (class_names[y_true_run[idx]], class_names[y_pred_run[idx]])
    error_pairs[pair] += 1

print("\nTop Confusion Pairs:")
for pair, cnt in error_pairs.most_common(10):
    print(f"  {pair[0]:20s} → {pair[1]:20s} : {cnt}")

# Save error summary
error_df = pd.DataFrame([{
    'true_label':  class_names[y_true_run[i]],
    'pred_label':  class_names[y_pred_run[i]],
    'confidence':  wrong_conf[j],
    'filename':    test_filenames[i]
} for j, i in enumerate(wrong_idx)])
error_df.sort_values('confidence', ascending=False, inplace=True)
error_df.to_csv(os.path.join(RESULT_DIR, "error_analysis.csv"), index=False)
print(f"\nError analysis CSV saved ({len(error_df)} rows)")


In [ ]:
# ========== QUALITATIVE ANALYSIS: TOP-5 MISCLASSIFIED PER CONFUSION PAIR ========== #
def _gradcam(img_path, model, last_conv):
    img_pil  = tf.keras.preprocessing.image.load_img(img_path, target_size=INPUT_SHAPE[:2])
    img_arr  = tf.keras.preprocessing.image.img_to_array(img_pil)
    img_proc = inception_preprocess(np.expand_dims(img_arr.copy(), 0))
    preds    = model.predict(img_proc, verbose=0)
    pred_idx = np.argmax(preds[0])
    hm = make_gradcam_heatmap(img_proc, model, last_conv, pred_idx)
    hm_r = cv2.resize(hm, INPUT_SHAPE[:2][::-1])
    hm_c = cv2.applyColorMap(np.uint8(255 * hm_r), cv2.COLORMAP_JET)
    hm_c = cv2.cvtColor(hm_c, cv2.COLOR_BGR2RGB)
    sup  = np.clip((hm_c * 0.4 + img_arr) / (img_arr.max() + 1e-8), 0, 1)
    return img_arr.astype('uint8'), sup, preds[0]

for pair, cnt in error_pairs.most_common(6):
    true_cls, pred_cls = pair
    pair_idx = [i for i in wrong_idx
                if class_names[y_true_run[i]] == true_cls and class_names[y_pred_run[i]] == pred_cls][:5]
    if not pair_idx:
        continue
    n = len(pair_idx)
    fig, axes = plt.subplots(n, 2, figsize=(7, n * 2.2))
    if n == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(f'True: {true_cls}  →  Pred: {pred_cls}  ({cnt} errors)', fontweight='bold')
    for row, idx in enumerate(pair_idx):
        orig, sup, probs = _gradcam(test_filenames[idx], model, LAST_CONV)
        axes[row, 0].imshow(orig); axes[row, 0].axis('off')
        axes[row, 0].set_title(f'Conf={probs[np.argmax(probs)]:.3f}', fontsize=7)
        axes[row, 1].imshow(sup); axes[row, 1].axis('off')
        axes[row, 1].set_title('Grad-CAM', fontsize=7)
    plt.tight_layout()
    fname = f"qualitative_{true_cls.lower()}_as_{pred_cls.lower()}.png"
    plt.savefig(os.path.join(RESULT_DIR, fname), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")


# Section 4 — Model Profiling & Deployment Diagnostics

Detailed per-model statistics: parameter count, disk size, inference speed, Top-K accuracy, per-class breakdown, predictions CSV, and final export ZIP.


In [ ]:
# ========== MODEL SUMMARY & PARAMETER COUNT ========== #
total_params     = model.count_params()
trainable_params = sum([np.prod(v.shape) for v in model.trainable_weights])
frozen_params    = total_params - trainable_params

print("=" * 55)
print("MODEL PARAMETER SUMMARY — InceptionV3")
print("=" * 55)
print(f"  Total parameters    : {total_params:>12,}")
print(f"  Trainable parameters: {trainable_params:>12,}")
print(f"  Frozen parameters   : {frozen_params:>12,}")
print(f"  Trainable ratio     : {trainable_params/total_params*100:.1f}%")
print("=" * 55)
model.summary()


In [ ]:
# ========== MODEL SIZE ON DISK ========== #
temp_model_path = "temp_inceptionv3.keras"
model.save(temp_model_path)
size_bytes = os.path.getsize(temp_model_path)
size_mb    = size_bytes / (1024 ** 2)
os.remove(temp_model_path)
print(f"Model disk size: {size_mb:.2f} MB  ({size_bytes:,} bytes)")

params_df = pd.DataFrame([{
    'Total_Params': total_params,
    'Trainable_Params': trainable_params,
    'Frozen_Params': frozen_params,
    'Trainable_Ratio_%': round(trainable_params/total_params*100, 2),
    'Model_Size_MB': round(size_mb, 2)
}])
params_df.to_csv(os.path.join(BASE_RESULT_DIR, "model_params.csv"), index=False)
print("Model params CSV saved!")


In [ ]:
# ========== INFERENCE SPEED BENCHMARK ========== #
import time

# Warm-up
for imgs, _ in test_ds.take(2):
    _ = model.predict(imgs, verbose=0)

# Benchmark
n_trials = 5
times_per_trial = []
total_samples   = 0

for _ in range(n_trials):
    t0 = time.perf_counter()
    count = 0
    for imgs, _ in test_ds:
        _ = model.predict(imgs, verbose=0)
        count += imgs.shape[0]
    t1 = time.perf_counter()
    times_per_trial.append(t1 - t0)
    total_samples = count

avg_total   = np.mean(times_per_trial)
ms_per_img  = avg_total / total_samples * 1000
fps         = total_samples / avg_total

print("=" * 50)
print(f"INFERENCE SPEED — InceptionV3")
print("=" * 50)
print(f"  Test samples   : {total_samples}")
print(f"  Avg total time : {avg_total:.3f} s  (over {n_trials} trials)")
print(f"  ms / image     : {ms_per_img:.3f} ms")
print(f"  Throughput     : {fps:.1f} FPS")
print("=" * 50)

speed_df = pd.DataFrame([{'ms_per_image': round(ms_per_img, 3), 'FPS': round(fps, 1),
                           'total_test_samples': total_samples}])
speed_df.to_csv(os.path.join(BASE_RESULT_DIR, "inference_speed.csv"), index=False)
print("Inference speed saved!")


In [ ]:
# ========== TOP-K ACCURACY ========== #
all_probs_list, all_true_list = [], []
for imgs, lbls in test_ds:
    probs = model.predict(imgs, verbose=0)
    all_probs_list.append(probs)
    all_true_list.extend(lbls.numpy())

all_probs_np = np.vstack(all_probs_list)
all_true_np  = np.array(all_true_list)

topk_results = {}
for k in [1, 2, 3]:
    top_k_preds = np.argsort(all_probs_np, axis=1)[:, -k:]
    correct_k   = sum(all_true_np[i] in top_k_preds[i] for i in range(len(all_true_np)))
    acc_k = correct_k / len(all_true_np)
    topk_results[f'Top-{k}'] = acc_k
    print(f"  Top-{k} Accuracy: {acc_k:.4f} ({correct_k}/{len(all_true_np)})")

topk_df = pd.DataFrame([topk_results])
topk_df.to_csv(os.path.join(BASE_RESULT_DIR, "topk_accuracy.csv"), index=False)
print("Top-K accuracy saved!")


In [ ]:
# ========== PER-CLASS METRICS CSV (BEST RUN) ========== #
from sklearn.metrics import classification_report

report_dict = classification_report(y_true_run, y_pred_run,
                                    target_names=class_names, output_dict=True)
per_class_rows = []
for cls in class_names:
    row = {'class': cls}
    row.update(report_dict[cls])
    per_class_rows.append(row)

per_class_csv = pd.DataFrame(per_class_rows)
per_class_csv.to_csv(os.path.join(RESULT_DIR, "per_class_metrics.csv"), index=False)
print("Per-class metrics CSV saved!")
print(per_class_csv.to_string(index=False))


In [ ]:
# ========== PREDICTIONS CSV (BEST RUN) ========== #
# Uses test_filenames collected from create_tf_datasets (avoids test_generator.filenames bug)
pred_probs = model.predict(test_ds, verbose=1)
pred_labels = np.argmax(pred_probs, axis=1)

predictions_df = pd.DataFrame({
    'filename':    test_filenames,
    'true_label':  [class_names[l] for l in test_labels_true],
    'pred_label':  [class_names[l] for l in pred_labels],
    'correct':     [t == p for t, p in zip(test_labels_true, pred_labels)],
    **{f'prob_{c}': pred_probs[:, i] for i, c in enumerate(class_names)}
})
predictions_df.to_csv(os.path.join(RESULT_DIR, "predictions.csv"), index=False)
print(f"Predictions CSV saved: {len(predictions_df)} rows")
acc_check = predictions_df['correct'].mean()
print(f"Accuracy check from CSV: {acc_check:.4f}")


In [ ]:
# ========== COMPREHENSIVE FINAL SUMMARY REPORT ========== #
final_report = [
    "=" * 70,
    "InceptionV3 - COMPREHENSIVE EXPERIMENT REPORT",
    "=" * 70,
    "",
    "ARCHITECTURE",
    "─" * 40,
    "  Backbone       : InceptionV3 (ImageNet pretrained)",
    f"  Input Shape    : {INPUT_SHAPE}",
    f"  Unfreeze Blocks: {INCEPTION_UNFREEZE_LAYERS}",
    "  Custom Head    : GAP → BN → Dense(128,relu,L2) → Dropout(0.5) → Softmax(3)",
    "",
    "TRAINING CONFIG",
    "─" * 40,
    f"  Seeds          : {SEEDS}",
    f"  Epochs         : {EPOCHS}",
    f"  Batch Size     : {BATCH_SIZE}",
    f"  Learning Rate  : {LEARNING_RATE} (ExponentialDecay)",
    f"  Label Smoothing: {LABEL_SMOOTHING}",
    "",
    "MULTI-RUN RESULTS (Mean ± Std)",
    "─" * 40,
]
for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    s = overall_stats[m]
    final_report.append(f"  {m:<15}: {s['mean']:.4f} ± {s['std']:.4f}")

final_report += [
    "",
    "ADVANCED METRICS (Mean ± Std)",
    "─" * 40,
    f"  Cohen's κ      : {np.mean(kappas):.4f} ± {np.std(kappas):.4f}",
    f"  MCC            : {np.mean(mccs):.4f} ± {np.std(mccs):.4f}",
    f"  Balanced Acc   : {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}",
    "",
    "MODEL PROFILING",
    "─" * 40,
    f"  Total Params   : {total_params:,}",
    f"  Trainable      : {trainable_params:,}  ({trainable_params/total_params*100:.1f}%)",
    f"  Disk Size      : {size_mb:.2f} MB",
    f"  Inference      : {ms_per_img:.3f} ms/image  |  {fps:.1f} FPS",
    "",
    "TOP-K ACCURACY (Best Run)",
    "─" * 40,
]
for k, v in topk_results.items():
    final_report.append(f"  {k:<8}: {v:.4f}")

final_report += ["", "=" * 70, "END OF REPORT", "=" * 70]

report_text_final = "\n".join(final_report)
print(report_text_final)
with open(os.path.join(BASE_RESULT_DIR, "COMPREHENSIVE_FINAL_REPORT.txt"), "w") as f:
    f.write(report_text_final)
print("\nFinal report saved!")


In [ ]:
# ========== LIST ALL OUTPUT FILES ========== #
print("=" * 60)
print("ALL OUTPUT FILES")
print("=" * 60)
for root, dirs, files in os.walk(BASE_RESULT_DIR):
    level = root.replace(BASE_RESULT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = ' ' * 2 * (level + 1)
    for f in sorted(files):
        fpath = os.path.join(root, f)
        size  = os.path.getsize(fpath)
        print(f"{sub_indent}{f:<45} {size:>10,} bytes")
print("=" * 60)


In [ ]:
# ========== ZIP SINGLE-RUN COMPLETE REPORT ========== #
import shutil
zip_output_path = "/kaggle/working/InceptionV3_Complete_Report"
shutil.make_archive(zip_output_path, 'zip', BASE_RESULT_DIR)
print(f"Complete report zipped → {zip_output_path}.zip")
from IPython.display import FileLink
display(FileLink(f'{zip_output_path}.zip'))
